# FinGPT — Resume Backtest from Agent 1 CSV Cache

## Workflow
```
CSV (Agent 1 cache)
        │
        ▼
Reconstruct NewsFingerprint objects (ticker + sentiment from CSV)
        │
        ▼  [optional]
Merge with HuggingFace dataset for full article texts
        │
        ▼
Agent 2 — vLLM real logprobs  (A/B/C → BUY/HOLD/SELL)
   2 batched engine.generate() calls per batch of 10
        │
        ▼
Price fetch — yfinance daily bars  (1d interval, close-to-close return)
        │
        ▼
Metrics + new CSV
```

## Key fixes applied vs the original run
| Bug | Root cause | Fix |
|-----|-----------|-----|
| HOLD always wins | `"Strategy: "` prefix has strong LM prior for word "HOLD" | Score letters **A/B/C** instead — equal prior, context-discriminative |
| All price fetches fail | `interval="1wk"` over a 7-day window → 1 bar, need ≥2 | Switch to `interval="1d"` (5 trading days per window) |

In [1]:
# ── 1. Install dependencies (Colab) ──────────────────────────────────────────
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

_pip("vllm")                                # vLLM inference engine
_pip("transformers", "accelerate")          # tokenizer + model loading
_pip("datasets")                            # HuggingFace datasets (optional merge)
_pip("yfinance", "pandas")                  # price fetching + data wrangling
_pip("python-dotenv", "pydantic")           # config + schema
print("Dependencies installed.")

Dependencies installed.


In [ ]:
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab — skipping Drive mount.")

REPO_DIR = "/content/FinGPT_Part2"          # change if you placed it elsewhere

if IN_COLAB and not os.path.exists(REPO_DIR):

    DRIVE_REPO = "/content/drive/MyDrive/FinGPT_Part2"  
    if os.path.exists(DRIVE_REPO):
        import shutil
        shutil.copytree(DRIVE_REPO, REPO_DIR)
    else:
        raise FileNotFoundError(
            f"Repo not found at {DRIVE_REPO}. "
            "Please clone it or adjust DRIVE_REPO above."
        )
elif not IN_COLAB:
    # Local development — point at the actual workspace
    REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), ""))
    if not os.path.exists(os.path.join(REPO_DIR, "config.py")):
        REPO_DIR = os.getcwd()              # fallback: current directory

# Add repo root to Python path
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repo dir : {REPO_DIR}")
print(f"sys.path : {sys.path[:3]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo dir : /content/FinGPT_Part2
sys.path : ['/content/FinGPT_Part2', '/content', '/env/python']


In [ ]:
import os

# Setting VLLM_ENABLE_V1_MULTIPROCESSING=0 forces the v1 engine to run
# in-process (InprocClient) — no subprocess, no fileno call.
# Must be set BEFORE LLM() is called (vLLM reads it lazily at init time).
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# Path to the DeepSeek-R1 / FinGPT model weights
os.environ["FINGPT_MODEL_PATH"] = '/content/drive/MyDrive/deepseek_fingpt_outputs/merged_for_vllm'

# Share one vLLM engine between both agents (saves GPU memory)
os.environ["SHARE_SINGLE_LLM_BETWEEN_AGENTS"] = "1"

# Use a FRESH yfinance cache so stale None values from the previous failed run
# don't shadow the corrected daily-interval fetches.
os.environ["FINGPT_YF_CACHE_PATH"] = "/tmp/yfinance_return_cache_resume.json"

# Softmax temperature for calibrating logprobs → probabilities
os.environ["FINGPT_CALIBRATION_T"] = "1.2"

# CoT token budget for Agent 2
os.environ["FINGPT_LOGITS_MAX_TOKENS"] = "1024"

print("Environment configured.")
print(f"  VLLM_ENABLE_V1_MULTIPROCESSING = {os.environ['VLLM_ENABLE_V1_MULTIPROCESSING']}")
print(f"  FINGPT_MODEL_PATH              = {os.environ['FINGPT_MODEL_PATH']}")

Environment configured.
  VLLM_ENABLE_V1_MULTIPROCESSING = 0
  FINGPT_MODEL_PATH              = /content/drive/MyDrive/deepseek_fingpt_outputs/merged_for_vllm


In [ ]:
# ── 4. Load Agent 1 result CSV (the cache) ────────────────────────────────────
import ast
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/backtest_20260501T015749Z.csv"

raw_df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(raw_df)} rows from CSV.")
print(f"Columns: {list(raw_df.columns)}")
print()

agent1_ok = raw_df[raw_df["skipped_reason"] != "fingerprint_failed"].copy()
print(f"Agent 1 successes: {len(agent1_ok)} / {len(raw_df)}")

def _parse_probs(val):
    if pd.isna(val) or not isinstance(val, str):
        return {"POSITIVE": 1/3, "NEGATIVE": 1/3, "NEUTRAL": 1/3}
    try:
        return ast.literal_eval(val)
    except Exception:
        return {"POSITIVE": 1/3, "NEGATIVE": 1/3, "NEUTRAL": 1/3}

agent1_ok["sentiment_probabilities_parsed"] = agent1_ok["sentiment_probabilities"].apply(_parse_probs)

display(agent1_ok[["ticker", "start_date", "end_date", "sentiment_label",
                    "sentiment_confidence", "skipped_reason"]].head(5))

Loaded 300 rows from CSV.
Columns: ['ticker', 'start_date', 'end_date', 'article_text', 'fingpt_label', 'sentiment_label', 'sentiment_confidence', 'sentiment_probabilities', 'signal_direction', 'signal_confidence', 'signal_strategy_tag', 'signal_logits', 'signal_probabilities', 'realized_return', 'position', 'strategy_return', 'skipped_reason']

Agent 1 successes: 291 / 300


,ticker,start_date,end_date,sentiment_label,sentiment_confidence,skipped_reason
0,AXP,2024-02-11,2024-02-18,POSITIVE,0.887585,price_fetch_failed
1,AXP,2024-02-18,2024-02-25,POSITIVE,0.966173,price_fetch_failed
2,AXP,2024-03-03,2024-03-10,POSITIVE,0.952435,price_fetch_failed
3,AXP,2024-03-03,2024-03-10,POSITIVE,0.966976,price_fetch_failed
4,AXP,2024-03-17,2024-03-24,POSITIVE,0.976539,price_fetch_failed


In [ ]:
FULL_TEXT_MAP: dict[tuple, str] = {}

HF_DATASET_NAME = "Piyush5911/FinGPT_Sentiment"

try:
    from datasets import load_dataset as hf_load_dataset
    from backtest.dataset_parser import build_backtest_rows, load_dataset as _load_ds

    hf_df = _load_ds(HF_DATASET_NAME)
    from backtest.dataset_parser import build_backtest_rows
    hf_rows = build_backtest_rows(hf_df)

    for r in hf_rows:
        key = (r["ticker"], r["start_date"])
        if key not in FULL_TEXT_MAP and r.get("article_text"):
            FULL_TEXT_MAP[key] = r["article_text"]

    print(f"Full-text map built: {len(FULL_TEXT_MAP)} entries.")
except Exception as exc:
    print(f"[optional] Could not load HF dataset: {exc}")
    print("Proceeding with 120-char truncated article texts from the CSV.")

[optional] Could not load HF dataset: Dataset 'Piyush5911/FinGPT_Sentiment' doesn't exist on the Hub or cannot be accessed.
Proceeding with 120-char truncated article texts from the CSV.


In [ ]:
from agent1.schema import NewsFingerprint

_LABEL_TO_SCORE = {"POSITIVE": 1.0, "NEGATIVE": -1.0, "NEUTRAL": 0.0}

fingerprints: list[NewsFingerprint] = []
fp_row_idx: list[int] = []

for idx, row in agent1_ok.iterrows():
    ticker      = str(row["ticker"])
    start_date  = str(row["start_date"])
    article_csv = str(row["article_text"]) if pd.notna(row["article_text"]) else ""

    # Prefer the full text from the HF dataset if available
    article_text = FULL_TEXT_MAP.get((ticker, start_date), article_csv)

    label = str(row.get("sentiment_label", "NEUTRAL"))
    if label not in _LABEL_TO_SCORE:
        label = "NEUTRAL"

    conf = float(row["sentiment_confidence"]) if pd.notna(row.get("sentiment_confidence")) else 0.5
    probs = row["sentiment_probabilities_parsed"]

    try:
        fp = NewsFingerprint(
            source="csv_cache",
            published_at=start_date,
            headline=article_text[:80],
            companies_named=[ticker],
            event_keywords=[],
            sentiment_label=label,
            sentiment_score=_LABEL_TO_SCORE[label],
            sentiment_confidence=conf,
            sentiment_probabilities=probs,
            calibration_T=1.2,
            article_text=article_text,
        )
        fingerprints.append(fp)
        fp_row_idx.append(idx)
    except Exception as exc:
        print(f"[row {idx}] Fingerprint construction failed: {exc}")

print(f"Reconstructed {len(fingerprints)} fingerprints from {len(agent1_ok)} Agent 1 rows.")

Reconstructed 291 fingerprints from 291 Agent 1 rows.


In [ ]:
import os, sys
from vllm import LLM
from transformers import AutoTokenizer
from agent2.reasoner import set_shared_vllm_engine
import agent2.reasoner as _a2

MODEL_PATH = os.environ["FINGPT_MODEL_PATH"]

_nb_stdout = sys.stdout
try:
    _real_fd = os.dup(1)
    sys.stdout = os.fdopen(_real_fd, 'w', buffering=1)
except Exception:
    pass

print(f"Loading vLLM engine from: {MODEL_PATH}")
try:
    engine = LLM(
        model=MODEL_PATH,
        trust_remote_code=True,
        dtype="auto",
        gpu_memory_utilization=0.85,
        disable_log_stats=True,
        enforce_eager=True,
    )
finally:
    # Restore notebook stdout so subsequent cells print normally.
    sys.stdout = _nb_stdout

print("vLLM engine loaded.")

# Inject engine and tokenizer into Agent 2 module directly
set_shared_vllm_engine(engine)
_a2._chat_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("Engine and tokenizer injected into Agent 2.")

INFO 05-01 03:20:50 [utils.py:233] non-default args: {'trust_remote_code': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/content/drive/MyDrive/deepseek_fingpt_outputs/merged_for_vllm'}
WARNING 05-01 03:20:50 [envs.py:1818] Unknown vLLM environment variable detected: VLLM_USE_V1
WARNING 05-01 03:20:50 [arg_utils.py:1467] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 05-01 03:20:50 [model.py:555] Resolved architecture: LlamaForCausalLM
INFO 05-01 03:20:50 [model.py:1680] Using max model len 131072
WARNING 05-01 03:20:50 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-01 03:20:50 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilati

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 05-01 03:24:06 [default_loader.py:384] Loading weights took 191.83 seconds
INFO 05-01 03:24:06 [gpu_model_runner.py:4879] Model loading took 14.99 GiB memory and 193.513875 seconds
INFO 05-01 03:24:20 [gpu_worker.py:440] Available KV cache memory: 17.62 GiB
INFO 05-01 03:24:20 [kv_cache_utils.py:1711] GPU KV cache size: 144,320 tokens
INFO 05-01 03:24:20 [kv_cache_utils.py:1716] Maximum concurrency for 131,072 tokens per request: 1.10x
INFO 05-01 03:24:20 [core.py:306] init engine (profile, create kv cache, warmup model) took 13.47 s
vLLM engine loaded.
Engine and tokenizer injected into Agent 2.


In [ ]:

from agent2.reasoner import generate_signal_batch
from agent2.schema import TradingSignal
from typing import Optional

BATCH_SIZE = 10

signals: list[Optional[TradingSignal]] = []
total = len(fingerprints)

for batch_start in range(0, total, BATCH_SIZE):
    batch_fps = fingerprints[batch_start : batch_start + BATCH_SIZE]
    batch_end = batch_start + len(batch_fps)
    print(f"Agent 2 batch {batch_start + 1}–{batch_end} / {total} ...", end=" ", flush=True)
    try:
        batch_signals = generate_signal_batch(batch_fps)
    except Exception as exc:
        print(f"FAILED: {exc}")
        batch_signals = [None] * len(batch_fps)
    signals.extend(batch_signals)
    n_ok = sum(s is not None for s in batch_signals)
    print(f"ok={n_ok}/{len(batch_fps)}")

assert len(signals) == len(fingerprints)

# Quick distribution check
from collections import Counter
direction_counts = Counter(
    (s.direction if s else "None") for s in signals
)
print(f"\nSignal direction distribution: {dict(direction_counts)}")

Agent 2 batch 1–10 / 291 ... 

Rendering prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/30 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/30 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

ok=1/1

Signal direction distribution: {'neutral': 278, 'short': 13}


In [ ]:

from backtest.price_fetcher import get_realized_return

realized_returns: list[Optional[float]] = []
fetch_ok = 0

for i, idx in enumerate(fp_row_idx):
    row = agent1_ok.loc[idx]
    ret = get_realized_return(
        ticker=str(row["ticker"]),
        start_date=str(row["start_date"]),
        end_date=str(row["end_date"]),
    )
    realized_returns.append(ret)
    if ret is not None:
        fetch_ok += 1
    if (i + 1) % 50 == 0:
        print(f"Price fetch progress: {i+1}/{len(fp_row_idx)} (ok={fetch_ok})")

print(f"\nPrice fetch complete: {fetch_ok}/{len(fp_row_idx)} returned a valid return.")

Price fetch progress: 50/291 (ok=0)
Price fetch progress: 100/291 (ok=0)
Price fetch progress: 150/291 (ok=0)
Price fetch progress: 200/291 (ok=0)
Price fetch progress: 250/291 (ok=0)


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WBA']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WBA']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WBA']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WBA']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WBA']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WBA']: YFTzMissingError('possibly delisted; no timezone found')



Price fetch complete: 0/291 returned a valid return.


In [ ]:
import math
from backtest.price_fetcher import direction_from_return

def _position_from_direction(direction: str) -> int:
    return {"long": 1, "short": -1}.get(direction, 0)

rows_out = []

for i, (df_idx, fp, sig, ret) in enumerate(
    zip(fp_row_idx, fingerprints, signals, realized_returns)
):
    orig = agent1_ok.loc[df_idx]
    record = {
        "ticker": orig["ticker"],
        "start_date": orig["start_date"],
        "end_date": orig["end_date"],
        "article_text": str(fp.article_text)[:120],
        "fingpt_label": orig["fingpt_label"],
        # Agent 1
        "sentiment_label": fp.sentiment_label,
        "sentiment_confidence": fp.sentiment_confidence,
        "sentiment_probabilities": str(fp.sentiment_probabilities),
        # Agent 2 (freshly generated)
        "signal_direction": sig.direction if sig else None,
        "signal_confidence": sig.confidence if sig else None,
        "signal_strategy_tag": sig.strategy_tag if sig else None,
        "signal_logits": str(sig.signal_logits) if sig else None,
        "signal_probabilities": str(sig.signal_probabilities) if sig else None,
        # Backtest
        "realized_return": ret,
        "position": _position_from_direction(sig.direction) if sig else None,
        "strategy_return": _position_from_direction(sig.direction) * ret
                            if sig and ret is not None else None,
        "skipped_reason": "" if (sig and ret is not None)
                            else ("signal_failed" if not sig else "price_fetch_failed"),
    }
    rows_out.append(record)

results_df = pd.DataFrame(rows_out)
─
from backtest.backtester import compute_metrics
metrics = compute_metrics(results_df)

print("=" * 50)
print("BACKTEST METRICS")
print("=" * 50)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<28}: {v:.4f}")
    else:
        print(f"  {k:<28}: {v}")
print()

# Signal direction distribution
print("Signal direction breakdown:")
print(results_df["signal_direction"].value_counts(dropna=False).to_string())

BACKTEST METRICS
  total_rows                  : 291
  successful_rows             : 0
  skip_rate                   : 1.0000
  direction_accuracy          : 0.0000
  long_accuracy               : 0.0000
  short_accuracy              : 0.0000
  mean_strategy_return        : 0.0000
  std_strategy_return         : 0.0000
  annualized_sharpe           : 0.0000
  total_pnl                   : 0.0000
  vs_fingpt_accuracy          : 0.0000

Signal direction breakdown:
signal_direction
neutral    278
short       13


In [ ]:
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT_DIR = os.path.join(REPO_DIR, "output")
os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, f"backtest_resume_{timestamp}.csv")

results_df.to_csv(out_path, index=False)
print(f"Saved {len(results_df)} rows → {out_path}")

if IN_COLAB:
    drive_out = f"/content/drive/MyDrive/backtest_resume_{timestamp}.csv"
    results_df.to_csv(drive_out, index=False)
    print(f"Also saved to Drive → {drive_out}")

Saved 291 rows → /content/FinGPT_Part2/output/backtest_resume_20260501T033648Z.csv
Also saved to Drive → /content/drive/MyDrive/backtest_resume_20260501T033648Z.csv
